# ConsDB for LSSTCam — Visit Metadata and Observing Conditions

- **Creation date:** 2025-04-18  
- **Last update:** 2026-03-09  
- **Author:** Sylvie Dagoret-Campagne  
- **Environment:** USDF RSP — `LSST` kernel

This notebook connects to the Consolidated Database (ConsDB) to retrieve  
LSSTCam visit metadata and observing-condition telemetry.  
It produces Mollweide sky maps (with the Galactic plane and Deep Drilling Fields)  
and time-series plots of key atmospheric parameters.

**References:**
- ConsDB API: https://usdf-rsp-dev.slac.stanford.edu/consdb/  
- SDM schema browser: https://sdm-schemas.lsst.io/  
- ConsDB documentation: https://consdb.lsst.io/index.html


> Nightly validation collection pattern:  
> `LSSTCam/runs/nightlyValidation/{day_obs}/<lsst_distrib_tag>/DM-50157`


In [ ]:
from lsst.summit.utils import ConsDbClient

In [ ]:
import numpy as np

import matplotlib.pyplot as plt
import matplotlib.dates as mdates

import matplotlib as mpl
import matplotlib.cm as cm
import matplotlib.colors as colors
import matplotlib.cm as cmx
from matplotlib.colors import ListedColormap

import seaborn as sns

%matplotlib widget

from lsst.meas.algorithms.installGaussianPsf import FwhmPerSigma

from tqdm.notebook import tqdm

In [ ]:
# xx-small
# x-small
# small
# medium
# large
# x-large
# xx-large

plt.rcParams["figure.figsize"] = (10, 6)
plt.rcParams["axes.labelsize"] = "x-large"
plt.rcParams["axes.titlesize"] = "x-large"
plt.rcParams["xtick.labelsize"] = "x-large"
plt.rcParams["ytick.labelsize"] = "x-large"

In [ ]:
from astropy.table import Table, join
from astropy.time import Time

from astropy.coordinates import SkyCoord
from astropy import coordinates
import astropy.coordinates as coord
import astropy.units as u

In [ ]:
import lsst.geom as geom
import lsst
from lsst.geom import Angle

In [ ]:
# https://pipelines.lsst.io/modules/lsst.geom/getting-started.html
func_degToRad = lambda x: Angle(x, lsst.geom.degrees).asRadians()
func_zendtoAirmass = lambda x: 1.0 / np.cos(func_degToRad(x))
# func_wrap = lambda x : Angle(x,lsst.geom.radians).wrap(180.*lsst.geom.degrees)

In [ ]:
def ra_to_mollweide(ra_rad):
    """
    Transforme une ascension droite (RA en radians) pour une projection Mollweide :
    - RA doit être d'abord dans [0, 2π]
    - Puis ramené dans [-π, +π] pour correspondre au domaine Mollweide
    - Puis inversé pour que RA croisse vers la gauche sur la carte

    Paramètre:
    - ra_rad : array ou scalaire de RA en radians

    Retourne:
    - RA transformé pour projection Mollweide

    Discussion avec ChatGPT verifiée le 28/04/2025
    """

    # 1) Assurer que RA est entre 0 et 2π
    ra_rad = np.remainder(ra_rad + 2 * np.pi, 2 * np.pi)

    # 2) Remettre RA entre -π et +π
    ra_rad[ra_rad > np.pi] -= 2 * np.pi

    # 3) Inverser RA pour que RA croisse vers la gauche
    return -ra_rad

## 1. Configuration

Set the instrument name and configure the ConsDB connection.  
The proxy must be bypassed so that requests to `.consdb` are routed correctly.


In [ ]:
instrument = "LSSTCam"

In [ ]:
os.environ["no_proxy"] += ",.consdb"

In [ ]:
url = "http://consdb-pq.consdb:8080/consdb"

In [ ]:
consdb = ConsDbClient(url)

https://sdm-schemas.lsst.io/cdb_lsstcomcam.html#exposure

In [ ]:
# Query both consDB tables
# exposure = consdb.query("SELECT * FROM cdb_lsstcam.exposure WHERE science_program = 'BLOCK-351'")
# visits = consdb.query("SELECT * FROM cdb_lsstcam.visit1 WHERE science_program = 'BLOCK-351'")
# visits_ql = consdb.query("SELECT * FROM cdb_lsstcam.visit1_quicklook")
# visits_ql = consdb.query("SELECT * FROM cdb_lsstcam.visit1")

exposure = consdb.query("SELECT * FROM cdb_lsstcam.exposure WHERE day_obs >= 20250415")
visits = consdb.query("SELECT * FROM cdb_lsstcam.visit1 WHERE day_obs >= 20250415")
# visits_ql = consdb.query("SELECT * FROM cdb_lsstcam.visit1_quicklook")
visits_ql = consdb.query("SELECT * FROM cdb_lsstcam.visit1")

# Join using astropy's join function on 'visit_id'
exposure_join = exposure.rename_column("exposure_id", "visit_id")
merged_exposure = join(exposure, visits, keys="visit_id", join_type="inner")
merged_visits = join(visits, visits_ql, keys="visit_id", join_type="inner")

# Display or use the merged table
# print(merged_visits)

## 3. Explore visit metadata

Inspect the column names, unique observation dates, filter names, science programs,  
and observation reasons present in the retrieved data.


In [ ]:
print(visits.columns)

In [ ]:
df_visits = visits.to_pandas()

### Observation dates

In [ ]:
df_visits["day_obs"].unique()

### Filters

In [ ]:
print(df_visits["physical_filter"].unique(), df_visits["band"].unique())

#### Remove bad filters

In [ ]:
df_visits = df_visits[
    ~(
        (df_visits["physical_filter"] == "other")
        | (df_visits["physical_filter"] == "none")
        | df_visits["physical_filter"]
        == "other:pinhole"
    )
]

### Remove rows with bad ra and dec

In [ ]:
df_visits = df_visits.dropna(subset=["s_ra", "s_dec"])

### Science program and observation reason

In [ ]:
df_visits["science_program"].unique()

In [ ]:
df_visits["observation_reason"].unique()

### Time exposure

In [ ]:
fig, ax = plt.subplots(1, 1, figsize=(8, 4))
ax.hist(df_visits["exp_time"].values, bins=50, facecolor="b")
plt.show()

## 4. Sky coverage maps (Mollweide projection)

Display LSSTCam visit positions in the Mollweide equal-area projection.  
The colour encodes the time elapsed since the first observation (in days).  
The `ra_to_mollweide()` helper converts RA to the axis convention required  
by matplotlib (RA = 0h at centre, increasing to the left).


In [ ]:
# dict_b_to_col = {"u": "b", "g":"g", "r": "r", "i": 'orange', 'z': "magenta",'y':'purple'}
dict_b_to_col = {
    "u": "b",
    "g": "g",
    "r": "r",
    "i": "orange",
    "z": "magenta",
    "y": "purple",
    "other": "grey",
    "none": "k",
    "other:pinhole": "grey",
}

In [ ]:
col = df_visits["band"].map(lambda b: dict_b_to_col[b]).values

In [ ]:
col

In [ ]:
palette_spectral = sns.color_palette("Spectral_r", as_cmap=True)
cmap_time = ListedColormap(sns.color_palette("Spectral_r", df_visits.size))
dt = df_visits["obs_start_mjd"].values - df_visits["obs_start_mjd"].min()
dtmin = dt.min()
dtmax = dt.max()

In [ ]:
fig, ax = plt.subplots(1, 1, figsize=(12, 6))
im = df_visits.plot.scatter(x="s_ra", y="s_dec", ax=ax, c=dt, cmap=cmap_time)
ax.grid()
ax.set_aspect("equal")
# cbar = plt.colorbar(im, orientation='horizontal', label='time(days)')
plt.show()

In [ ]:
palette_spectral = sns.color_palette("Spectral_r", as_cmap=True)

In [ ]:
cmap_time = ListedColormap(sns.color_palette("Spectral", df_visits.size))

In [ ]:
dt = df_visits["obs_start_mjd"].values - df_visits["obs_start_mjd"].min()

In [ ]:
dtmin = dt.min()
dtmax = dt.max()

In [ ]:
ra_g = df_visits["s_ra"].apply(func_degToRad)
dec_g = df_visits["s_dec"].apply(func_degToRad)

In [ ]:
# Galactic plane
gal_long = np.linspace(-180.0, 180, 360)
gal_lat = np.zeros((360))
coordinates_galactic_planes = SkyCoord(l=gal_long * u.degree, b=gal_lat * u.degree, frame="galactic")
gp_radec = coordinates_galactic_planes.transform_to("icrs")

# for galactic plane
# gp_radec.ra : 0., 360.
# gp_radec.dec : -90, 90
# Old method
# gp_ra_toplot = coordinates.Angle(gp_radec.ra.degree*u.degree)
# gp_ra_toplot = gp_ra_toplot.wrap_at(180*u.degree)
# gp_ra_toplot -180, 180

# new method in radian with ra_to_mollweide
gp_ra_toplot = ra_to_mollweide(gp_radec.ra.radian)
gp_dec_toplot = gp_radec.dec.radian

#### Convert data (ra,dec) into radian

In [ ]:
ra = coordinates.Angle(df_visits["s_ra"].values, unit="deg").radian
dec = coordinates.Angle(df_visits["s_dec"].values, unit="deg").radian

In [ ]:
# plot
# x arg in rad must be (-2pi,2pi), y arg in rad  must be in ( -pi,pi )
fig = plt.figure(figsize=(12, 6))
# ax = fig.add_subplot(111, projection="aitoff")
ax = fig.add_subplot(111, projection="mollweide")

# galactic plane
ax.scatter(gp_ra_toplot, gp_dec_toplot, c="r", label="Galactic Plane", s=1)

# ax.scatter(ra-np.pi,dec,marker='+',c=all_colors_g)
# im = ax.scatter(ra_g.values-np.pi,dec_g.values,marker='+',s=50,lw=3,c=dt,cmap=palette_spectral)
im = ax.scatter(
    ra_to_mollweide(ra_g.values), dec_g.values, marker="+", s=50, lw=3, c=dt, cmap=palette_spectral
)

ax.legend()
ax.grid()

cbar = plt.colorbar(im, orientation="horizontal", label="time(days)")
plt.suptitle(instrument, fontsize=20, fontweight="bold")
plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Coordonnées des Deep Drilling Fields (RA en degrés, Dec en degrés)
ddf_names = ["XMM-LSS", "COSMOS", "ECDFS", "ELAIS-S1", "EDFS_a", "EDFS_b"]
ddf_ra_deg = np.array([35.57, 150.11, 52.98, 9.45, 58.90, 63.60])  # approximatif
ddf_dec_deg = np.array([-4.82, 2.23, -28.12, -44.02, -49.32, -47.60])  # approximatif


# Conversion en radians et adaptation pour Mollweide projection
ddf_ra_rad = np.radians(ddf_ra_deg)
ddf_dec_rad = np.radians(ddf_dec_deg)
# Décalage pour Mollweide (RA=0h au centre), donc -π à π
# ddf_ra_rad = np.remainder(ddf_ra_rad + 2*np.pi, 2*np.pi)  # force RA entre 0 et 2pi
# ddf_ra_rad[ddf_ra_rad > np.pi] -= 2*np.pi  # force entre -pi et +pi
# ddf_ra_rad = -ddf_ra_rad  # reverse l'axe des RA
ddf_ra_rad = ra_to_mollweide(ddf_ra_rad)

# --- Ton plot existant ---
fig = plt.figure(figsize=(12, 6))
ax = fig.add_subplot(111, projection="mollweide")

# Exemple de paln galactique
ax.scatter(gp_ra_toplot, gp_dec_toplot, c="r", label="Galactic Plane", s=1)
# im = ax.scatter(ra_g.values - np.pi, dec_g.values, marker='+', s=50, lw=3, c=dt, cmap=palette_spectral)
im = ax.scatter(
    ra_to_mollweide(ra_g.values), dec_g.values, marker="+", s=50, lw=2, c=dt, cmap=palette_spectral
)

# Ajout des Deep Drilling Fields
for i in range(len(ddf_names)):
    ax.plot(ddf_ra_rad[i], ddf_dec_rad[i], "o", markersize=10, color="gold", markeredgecolor="black")
    ax.text(
        ddf_ra_rad[i] + 0.05,
        ddf_dec_rad[i] + 0.05,
        ddf_names[i],
        fontsize=10,
        color="gold",
        ha="left",
        va="bottom",
        weight="bold",
    )

ax.legend()
ax.grid()

cbar = plt.colorbar(im, orientation="horizontal", label="time (days)")
plt.suptitle(instrument, fontsize=20, fontweight="bold")
plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Coordonnées des Deep Drilling Fields (RA en degrés, Dec en degrés)
ddf_names = ["XMM-LSS", "COSMOS", "ECDFS", "ELAIS-S1", "EDFS_a", "EDFS_b"]
ddf_ra_deg = np.array([35.57, 150.11, 52.98, 9.45, 58.90, 63.60])  # approximatif
ddf_dec_deg = np.array([-4.82, 2.23, -28.12, -44.02, -49.32, -47.60])  # approximatif


# Conversion en radians et adaptation pour Mollweide projection
ddf_ra_rad = np.radians(ddf_ra_deg)
ddf_dec_rad = np.radians(ddf_dec_deg)
# ddf_ra_rad = np.remainder(ddf_ra_rad + 2*np.pi, 2*np.pi)
# ddf_ra_rad[ddf_ra_rad > np.pi] -= 2*np.pi
# ddf_ra_rad = -ddf_ra_rad
ddf_ra_rad = ra_to_mollweide(ddf_ra_rad)

# Rayon du cercle en radians (~3.5 degrés)
radius_deg = 3.5
radius_rad = np.radians(radius_deg)

# --- Ton plot existant ---
fig = plt.figure(figsize=(12, 6))
ax = fig.add_subplot(111, projection="mollweide")

# Plan galactique
ax.scatter(gp_ra_toplot, gp_dec_toplot, c="r", label="Galactic Plane", s=1)
# im = ax.scatter(ra_g.values - np.pi, dec_g.values, marker='+', s=50, lw=3, c=dt, cmap=palette_spectral)
im = ax.scatter(
    ra_to_mollweide(ra_g.values), dec_g.values, marker="+", s=50, lw=3, c=dt, cmap=palette_spectral
)

# Couleur choisie
ddf_color = "navy"  # bleu foncé

# Ajout des Deep Drilling Fields
for i in range(len(ddf_names)):
    # Marqueur central
    ax.plot(ddf_ra_rad[i], ddf_dec_rad[i], "o", markersize=10, lw=2, color=ddf_color, markeredgecolor="black")
    ax.text(
        ddf_ra_rad[i] + 0.05,
        ddf_dec_rad[i] + 0.05,
        ddf_names[i],
        fontsize=10,
        color=ddf_color,
        ha="left",
        va="bottom",
        weight="bold",
    )

    # Cercle autour
    theta = np.linspace(0, 2 * np.pi, 100)
    circle_ra = ddf_ra_rad[i] + radius_rad * np.cos(theta)
    circle_dec = ddf_dec_rad[i] + radius_rad * np.sin(theta)
    ax.plot(circle_ra, circle_dec, color=ddf_color, linestyle="--", lw=1)

ax.legend()
ax.grid()

cbar = plt.colorbar(im, orientation="horizontal", label="time (days)")
plt.suptitle(instrument)

# --- Ajouter aussi le LMC ---
# Coordonnées du Grand Nuage de Magellan (LMC)
lmc_ra_deg = 80.8939
lmc_dec_deg = -69.7561

# Conversion en radians
lmc_ra_rad = np.radians(lmc_ra_deg)
lmc_dec_rad = np.radians(lmc_dec_deg)

# Ajustement Mollweide
lmc_ra_rad = np.remainder(lmc_ra_rad + 2 * np.pi, 2 * np.pi)
if lmc_ra_rad > np.pi:
    lmc_ra_rad -= 2 * np.pi
lmc_ra_rad = -lmc_ra_rad
# lmc_ra_rad = ra_to_mollweide([lmc_ra_rad])[0]

# Ajouter LMC au plot
ax.plot(
    lmc_ra_rad,
    lmc_dec_rad,
    marker="*",
    color="darkorange",
    markersize=15,
    markeredgecolor="black",
    label="LMC",
)
ax.text(
    lmc_ra_rad + 0.05,
    lmc_dec_rad + 0.05,
    "LMC",
    fontsize=10,
    color="darkorange",
    ha="left",
    va="bottom",
    weight="bold",
)

plt.suptitle(instrument, fontsize=20, fontweight="bold")
plt.show()

## 5. Observing conditions vs. time

Plot key telemetry quantities stored in ConsDB as a function of observation time:  
airmass, air temperature, pressure, humidity, wind speed / direction,  
DIMM seeing, and telescope focus position.


In [ ]:
time = Time(visits["exp_midpt"])

In [ ]:
print("tmin = ", time.min(), " | tmax = ", time.max())

### 5.1 Airmass vs. time

In [ ]:
# Plot the time vs the specified column
plt.figure(figsize=(12, 4))
plt.plot(time.plot_date, visits["airmass"], "b.")

# Set x-axis to show dates
# plt.gca().xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m-%d %H:%M:%S'))
plt.gca().xaxis.set_major_formatter(mdates.DateFormatter("%Y-%m-%d %H"))
plt.gca().xaxis.set_major_locator(mdates.AutoDateLocator())

# Rotate and format x-axis labels for readability
plt.xticks(rotation=45, ha="right")

plt.xlabel("Time")
plt.ylabel("Airmass")
plt.title(f"Time vs Airmass")
plt.legend()
plt.grid(True)
plt.tight_layout()  # Adjust layout to prevent clipping of labels
plt.show()

In [ ]:
# Plot the time vs the specified column
plt.figure(figsize=(12, 4))
plt.plot(time.plot_date, visits["air_temp"], "b.")

# Set x-axis to show dates
# plt.gca().xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m-%d %H:%M:%S'))
plt.gca().xaxis.set_major_formatter(mdates.DateFormatter("%Y-%m-%d %H"))
plt.gca().xaxis.set_major_locator(mdates.AutoDateLocator())

# Rotate and format x-axis labels for readability
plt.xticks(rotation=45, ha="right")

plt.xlabel("Time")
plt.ylabel("air_temp")
plt.title(f"air_temp vs time")
plt.legend()
plt.grid(True)
plt.tight_layout()  # Adjust layout to prevent clipping of labels
plt.show()

In [ ]:
# Plot the time vs the specified column
plt.figure(figsize=(12, 4))
plt.plot(time.plot_date, visits["pressure"], "b.")

# Set x-axis to show dates
# plt.gca().xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m-%d %H:%M:%S'))
plt.gca().xaxis.set_major_formatter(mdates.DateFormatter("%Y-%m-%d %H"))
plt.gca().xaxis.set_major_locator(mdates.AutoDateLocator())

# Rotate and format x-axis labels for readability
plt.xticks(rotation=45, ha="right")

plt.xlabel("Time")
plt.ylabel("pressure")
plt.title(f"pressure vs time")
plt.legend()
plt.grid(True)
plt.tight_layout()  # Adjust layout to prevent clipping of labels
plt.show()

In [ ]:
# Plot the time vs the specified column
plt.figure(figsize=(12, 4))
plt.plot(time.plot_date, visits["humidity"], "b.")

# Set x-axis to show dates
# plt.gca().xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m-%d %H:%M:%S'))
plt.gca().xaxis.set_major_formatter(mdates.DateFormatter("%Y-%m-%d %H"))
plt.gca().xaxis.set_major_locator(mdates.AutoDateLocator())

# Rotate and format x-axis labels for readability
plt.xticks(rotation=45, ha="right")

plt.xlabel("Time")
plt.ylabel("humidity")
plt.title(f"humidity vs time")
plt.legend()
plt.grid(True)
plt.tight_layout()  # Adjust layout to prevent clipping of labels
plt.show()

In [ ]:
# Plot the time vs the specified column
plt.figure(figsize=(12, 4))
plt.plot(time.plot_date, visits["wind_speed"], "b.")

# Set x-axis to show dates
# plt.gca().xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m-%d %H:%M:%S'))
plt.gca().xaxis.set_major_formatter(mdates.DateFormatter("%Y-%m-%d %H"))
plt.gca().xaxis.set_major_locator(mdates.AutoDateLocator())

# Rotate and format x-axis labels for readability
plt.xticks(rotation=45, ha="right")

plt.xlabel("Time")
plt.ylabel("wind_speed")
plt.title(f"wind_speed vs time")
plt.legend()
plt.grid(True)
plt.tight_layout()  # Adjust layout to prevent clipping of labels
plt.show()

In [ ]:
# Plot the time vs the specified column
plt.figure(figsize=(12, 4))
plt.plot(time.plot_date, visits["wind_dir"], "b.")

# Set x-axis to show dates
# plt.gca().xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m-%d %H:%M:%S'))
plt.gca().xaxis.set_major_formatter(mdates.DateFormatter("%Y-%m-%d %H"))
plt.gca().xaxis.set_major_locator(mdates.AutoDateLocator())

# Rotate and format x-axis labels for readability
plt.xticks(rotation=45, ha="right")

plt.xlabel("Time")
plt.ylabel("wind_dir")
plt.title(f"wind_dir vs time")
plt.legend()
plt.grid(True)
plt.tight_layout()  # Adjust layout to prevent clipping of labels
plt.show()

In [ ]:
# Plot the time vs the specified column
plt.figure(figsize=(12, 4))
plt.plot(time.plot_date, visits["dimm_seeing"], "b.")

# Set x-axis to show dates
# plt.gca().xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m-%d %H:%M:%S'))
plt.gca().xaxis.set_major_formatter(mdates.DateFormatter("%Y-%m-%d %H"))
plt.gca().xaxis.set_major_locator(mdates.AutoDateLocator())

# Rotate and format x-axis labels for readability
plt.xticks(rotation=45, ha="right")

plt.xlabel("Time")
plt.ylabel("dimm_seeing")
plt.title(f"dimm_seeing vs time")
plt.legend()
plt.grid(True)
plt.tight_layout()  # Adjust layout to prevent clipping of labels
plt.show()

In [ ]:
# Plot the time vs the specified column
plt.figure(figsize=(12, 4))
plt.plot(time.plot_date, visits["focus_z"], "b.")

# Set x-axis to show dates
# plt.gca().xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m-%d %H:%M:%S'))
plt.gca().xaxis.set_major_formatter(mdates.DateFormatter("%Y-%m-%d %H"))
plt.gca().xaxis.set_major_locator(mdates.AutoDateLocator())

# Rotate and format x-axis labels for readability
plt.xticks(rotation=45, ha="right")

plt.xlabel("Time")
plt.ylabel("focus_z")
plt.title(f"focus_z vs time")
plt.legend()
plt.grid(True)
plt.tight_layout()  # Adjust layout to prevent clipping of labels
plt.show()